In [5]:
import pandas as pd
from rdflib import Graph

def shorten_uri(uri):
    return str(uri).split("/")[-1].replace("_", " ")

#Runs a SPARQL query and stores the result as a pandas DataFrame. Also shortens URI columns if wanted
def query_to_dataframe(graph, query, shorten_columns=None):
    results = graph.query(query)

    rows = []
    for row in results:
        row_dict = {}

        for var, value in zip(results.vars, row):
            var_name = str(var)

            if value is None:
                row_dict[var_name] = None
            elif shorten_columns and var_name in shorten_columns:
                row_dict[var_name] = shorten_uri(value)
            else:
                row_dict[var_name] = value.toPython()

        rows.append(row_dict)

    return pd.DataFrame(rows)

In [6]:
# Load RDF graph
graph = Graph()
graph.parse("london_airbnb_kg.ttl", format="turtle")

print("Triples loaded:", len(graph))

Triples loaded: 1001479


In [31]:
get_all_boroughs = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT
    (?borough as ?name)
    ?airbnb_level
    ?housing_level
    ?transportation_level
WHERE {
    ?borough ex:hasPressureIndicator ?presureIndicator .
    ?presureIndicator ex:airbnbPressureLevel ?airbnb_level.

    ?borough ex:hasHousingIndicator ?housingIndicator .
    ?housingIndicator ex:housingPressureLevel ?housing_level.

    ?borough ex:hasTransportIndicator ?transportIndicator .
    ?transportIndicator ex:transportAccessibilityLevel ?transportation_level.
}
"""

result = query_to_dataframe(
    graph,
    get_all_boroughs,
    shorten_columns=["name", "pressure"]
)

pd.set_option('display.max_rows', None)
result

,name,airbnb_level,housing_level,transportation_level
0,Havering,Low,Low,Low
1,Bexley,Low,Low,Low
2,Sutton,Low,Low,Low
3,Barking and Dagenham,Low,Low,Low
4,City of London,High,Low,High
5,Harrow,Low,Medium,Low
6,Kingston upon Thames,Low,Low,Medium
7,Hillingdon,Low,Medium,Low
8,Enfield,Low,Low,Low
9,Bromley,Low,Low,Low


In [3]:
# Research Question 1
# Query 1: Find the boroughs with "High" airbnb pressure level and retrieve some of the borough-level housing or demographic indicators.
# Which indicators we should use for the "co-occurrence" analysis I don't fully know yet.
rq1_query1 = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT
    ?borough
    ?income
    ?housePrice
    ?popDensity
WHERE {
    ?borough ex:hasPressureIndicator ?pressure .

    ?pressure ex:airbnbPressureLevel ?level ;
              ex:airbnbPressureScore ?score .
    FILTER(str(?level) = "High")

    OPTIONAL { ?borough ex:medianIncome ?income . }
    OPTIONAL { ?borough ex:medianHousePrice ?housePrice . }
    OPTIONAL { ?borough ex:populationDensity ?popDensity . }
}
ORDER BY DESC(?score)
"""

rq1_query1_df = query_to_dataframe(
    graph,
    rq1_query1,
    shorten_columns=["borough"]
)

# Query 2: Compare the averages of some of the indicators between High pressure level boroughs and Low, Medium boroughs.
# The idea is to use these with some visualization (normal bar chart maybe?)
# to compare boroughs with different Airbnb pressure levels on their average borough-level housing or demographic indicators.
rq1_query2 = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT
    ?level
    (COUNT(?borough) AS ?boroughCount)
    (AVG(?score) AS ?avgPressureScore)
    (AVG(?income) AS ?avgIncome)
    (AVG(?housePrice) AS ?avgHousePrice)
    (AVG(?popDensity) AS ?avgPopDensity)
WHERE {
    ?borough ex:hasPressureIndicator ?pressure .

    ?pressure ex:airbnbPressureLevel ?level ;
              ex:airbnbPressureScore ?score .

    OPTIONAL { ?borough ex:medianIncome ?income . }
    OPTIONAL { ?borough ex:medianHousePrice ?housePrice . }
    OPTIONAL { ?borough ex:populationDensity ?popDensity . }
}
GROUP BY ?level
ORDER BY ?level
"""

rq1_query2_df = query_to_dataframe(graph, rq1_query2)

# Research Question 2
# Query 1: Examples of entire-home Airbnb listings in high Airbnb and housing pressure boroughs.
# Direct answer to RQ2 (with limit so that we don't print all thousands of them)
rq2_query1 = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT
    ?listing
    ?borough
    ?airbnbScore
    ?housingScore
    ?latitude
    ?longitude
WHERE {
    ?listing ex:isLocatedIn ?borough ;
             ex:hasRoomType ?roomType ;
             ex:latitude ?latitude ;
             ex:longitude ?longitude .

    ?roomType ex:roomTypeName ?roomName .
    FILTER(str(?roomName) = "Entire home/apt")

    ?borough ex:hasPressureIndicator ?pressure ;
             ex:hasHousingIndicator ?housing .

    ?pressure ex:airbnbPressureLevel ?airbnbLevel ;
              ex:airbnbPressureScore ?airbnbScore .
    FILTER(str(?airbnbLevel) = "High")

    ?housing ex:housingPressureLevel ?housingLevel ;
             ex:housingPressureScore ?housingScore .
    FILTER(str(?housingLevel) = "High")
}
"""

rq2_query1_df = query_to_dataframe(
    graph,
    rq2_query1,
    shorten_columns=["listing", "borough"]
)

# Query 2: Count the number of such listings per high pressure borough
# For further (bar chart) visualization alongside the London map dot plot idea.
rq2_query2 = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT
    ?borough
    (COUNT(?listing) AS ?listingCount)
WHERE {
    ?listing ex:isLocatedIn ?borough ;
             ex:hasRoomType ?roomType .

    ?roomType ex:roomTypeName ?roomName .
    FILTER(str(?roomName) = "Entire home/apt")

    ?borough ex:hasPressureIndicator ?pressure ;
             ex:hasHousingIndicator ?housing .

    ?pressure ex:airbnbPressureLevel ?airbnbLevel .
    FILTER(str(?airbnbLevel) = "High")

    ?housing ex:housingPressureLevel ?housingLevel .
    FILTER(str(?housingLevel) = "High")
}
GROUP BY ?borough
ORDER BY DESC(?listingCount)
"""

rq2_query2_df = query_to_dataframe(
    graph,
    rq2_query2,
    shorten_columns=["borough"]
)

# Research Question 3
# Query 1: Hosts with listings in more than one borough (with at least one of them a high pressure one).
# NOTE: City of London does not have pressure profiles (we still have to decide whether to include this).
rq3_query1 = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT
    ?host
    (COUNT(DISTINCT ?borough) AS ?boroughCount)
    (GROUP_CONCAT(DISTINCT STRAFTER(STR(?borough), "/borough/"); separator=", ") AS ?boroughs)
WHERE {
    ?host ex:hasListing ?listing .
    ?listing ex:isLocatedIn ?borough .

    ?borough ex:hasPressureIndicator ?pressure .
    ?pressure ex:airbnbPressureLevel ?pressureLevel .

    FILTER(BOUND(?pressureLevel))
}
GROUP BY ?host
HAVING (
    COUNT(DISTINCT ?borough) > 1 &&
    SUM(IF(str(?pressureLevel) = "High", 1, 0)) > 0
)
ORDER BY DESC(?boroughCount)
"""

rq3_query1_df = query_to_dataframe(
    graph,
    rq3_query1,
    shorten_columns=["host"]
)

# Research Question 4
# Query 1: Boroughs that fall into each profile based on Airbnb pressure, housing pressure and transport accessibility score.
# NOTE: Might be better to add the transport bands directly into the graphs.
rq4_query1 = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT
    ?pressureLevel
    ?housingLevel
    ?transportLevel
    (COUNT(?borough) AS ?boroughCount)
    (GROUP_CONCAT(DISTINCT STRAFTER(STR(?borough), "/borough/"); separator=", ") AS ?boroughs)
WHERE {
    ?borough ex:hasPressureIndicator ?pressure ;
             ex:hasHousingIndicator ?housing ;
             ex:hasTransportIndicator ?transport .

    ?pressure ex:airbnbPressureLevel ?pressureLevel .
    ?housing ex:housingPressureLevel ?housingLevel .
    ?transport ex:transportAccessibilityLevel ?transportLevel .
}
GROUP BY ?pressureLevel ?housingLevel ?transportLevel
ORDER BY DESC(?boroughCount)
"""

rq4_query1_df = query_to_dataframe(graph, rq4_query1)

# Query 2: Borough profile similarity
rq4_query2 = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT ?from_borough ?to_borough ?similarity
WHERE {
    ?profile ex:hasSource ?from_borough ;
             ex:hasTarget ?to_borough ;
             ex:similarityValue ?similarity .

    FILTER(?similarity > 0.8)
}
ORDER BY DESC(?similarity)
"""

rq4_query2_df = query_to_dataframe(
    graph,
    rq4_query2,
    shorten_columns=["from_borough", "to_borough"]
)

Triples loaded: 1001479


In [4]:
rq1_query1_df.to_csv("data/query_results/rq1_high_airbnb_pressure_boroughs.csv", index=False)
rq1_query2_df.to_csv("data/query_results/rq1_pressure_level_averages.csv", index=False)
rq2_query1_df.to_csv("data/query_results/rq2_entire_home_examples.csv", index=False)
rq2_query2_df.to_csv("data/query_results/rq2_entire_home_counts.csv", index=False)
rq3_query1_df.to_csv("data/query_results/rq3_multiborough_hosts.csv", index=False)
rq4_query1_df.to_csv("data/query_results/rq4_pressure_profile_clusters.csv", index=False)
rq4_query2_df.to_csv("data/query_results/rq4_similarity_pairs.csv", index=False)
